<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Abstract

## Abstract

FlyRank content teams may need to review many webpages for potential content refresh, making it useful to identify which pages should be investigated first. This study asks which webpages should be prioritized using existing search performance and content signals from an anonymized dataset of 30,000 webpages. We compare a simple freshness-and-visibility baseline with a Random Forest model using signals including impressions, average position, CTR, word count, content age, and days since the last update. The model and baseline are evaluated on the same client-holdout test set using Precision@20 and Precision@50 to measure ranking quality. The resulting ranked queue is intended to support human content-review decisions and is not presented as causal evidence or a prediction of Google's ranking algorithm.

# Introduction

## 1. Introduction and Problem Statement

A practical content challenge is deciding which webpages deserve attention first when a content team has more pages to review than it can manually investigate at once.

This project addresses that FlyRank content problem by creating a ranked review queue based on observable webpage and search-performance signals.

### Problem Statement

**Which webpages should be reviewed first for content refresh based on their existing search performance and content signals?**

The purpose of this project is not to predict Google's ranking algorithm. Instead, it is to provide decision support for a content team by identifying pages associated with an observed declining trend and placing higher-priority pages earlier in a review queue.

The analysis uses anonymized webpage-level observations and considers signals such as search impressions, average position, CTR, word count, content age, and days since the last update.

A simple rule-based baseline is compared with a Random Forest model using the same client-holdout evaluation design. The final model is then used to create a ranked list of pages for human investigation.

The findings are framed as observed and measured results within the available dataset. They do not establish that any individual feature causes a decline in search performance, and they should not be interpreted as predictions of Google's algorithm.

## 1. Question

*The research question and the decision it supports.*

## 1. Question

My research question is:

Which webpages should be reviewed first for content refresh based on their existing search performance and content signals?

The decision supported by this work is to help a content team decide which pages should receive attention first.

The goal is not to predict Google's algorithm. The goal is to provide a ranked list that can support human content-review decisions.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

This project uses the anonymized starter search dataset provided for the internship.

Each row represents one webpage.

The dataset contains search-performance and content-related information such as impressions, average position, CTR, word count, content age, and update age.

I use these fields because they are observable signals that can be available before making a content-refresh decision.

I exclude fields that directly reveal the outcome or could cause leakage, such as the trend percentage used to create the declining label.

The data is anonymized and does not contain client names, private URLs, or private search queries.

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 3. Methodology

I frame this as a ranking problem because the main goal is to decide which pages should be reviewed first.

The target is whether a page is declining. I define the label as 1 when trend_direction is "down" and 0 otherwise.

The baseline is a simple hand-written rule using page freshness and search visibility.

The model uses observable features such as content age, days since last update, impressions, average position, CTR, and word count.

I use a grouped/client-holdout validation approach where possible so that pages from the same client do not appear in both training and testing.

I also check for leakage and avoid using features that directly contain the outcome.

In [ ]:
# Create the target label

df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Declining pages:", df["is_declining"].sum())
print("Declining rate:", round(df["is_declining"].mean(), 3))

Declining pages: 16262
Declining rate: 0.542


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Interpretation

The model and baseline are evaluated on the same test set.

The comparison shows whether the learned model provides useful additional signal beyond the simple hand-written rule.

The result should be treated as measured decision-support performance on this dataset, not as proof that the model will perform the same way on every future dataset.

In [ ]:
import os
import pandas as pd
import numpy as np

# Find the repository
if os.path.exists("/content/flyrank-ml-internship-starter"):
    os.chdir("/content/flyrank-ml-internship-starter")

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# Features
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = df["is_declining"]

print("Everything is ready!")
print("Rows:", len(df))
print("Features:", features)
print("Declining rate:", round(y.mean(), 3))

Everything is ready!
Rows: 30000
Features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Declining rate: 0.542


## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

This project has several limitations.

First, the data is anonymized and represents a limited sample, so the results may not generalize to every website.

Second, the analysis shows associations in the available data. It does not prove that changing a page will cause its search performance to improve.

Third, the model cannot predict Google's ranking algorithm.

Fourth, the target label is based on observed trend direction, so it represents a defined outcome rather than a perfect measure of content quality.

Finally, model recommendations should be reviewed by a human before any content changes are made.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

# Train the model
model.fit(X, y)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
# Score all pages
df["refresh_score"] = model.predict_proba(X)[:, 1]

recommendations = df.sort_values(
    "refresh_score",
    ascending=False
).copy()

recommendations["recommended_action"] = np.where(
    recommendations["refresh_score"] >= 0.70,
    "High-priority content review",
    np.where(
        recommendations["refresh_score"] >= 0.40,
        "Review when capacity allows",
        "Monitor"
    )
)

columns_to_show = [
    "refresh_score",
    "recommended_action",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "content_age_days",
    "trend_direction"
]

display(recommendations[columns_to_show].head(20))

,refresh_score,recommended_action,impressions_90d,avg_position,ctr,days_since_last_update,content_age_days,trend_direction
3299,1.0,High-priority content review,19331,27.4,0.27,20,141,down
3296,1.0,High-priority content review,2731,2.2,0.04,104,144,down
2457,1.0,High-priority content review,4304,27.0,0.05,20,147,down
19576,1.0,High-priority content review,1748,2.2,0.06,104,330,down
14986,1.0,High-priority content review,13891,33.6,0.16,20,132,down
27217,1.0,High-priority content review,3690,3.7,0.22,20,103,down
15744,1.0,High-priority content review,1236,6.6,0.00,20,154,down
7053,1.0,High-priority content review,2294,35.9,0.00,104,284,down
7073,1.0,High-priority content review,102,12.2,0.00,104,223,down
25636,1.0,High-priority content review,3899,4.1,0.23,104,330,down


## 6. Ranked recommendations

The output is a ranked list of pages that may deserve content review.

Pages with higher scores are prioritized for human review.

The score should be interpreted as a decision-support signal rather than an automatic instruction to change a page.

A content specialist should check the page before taking action.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
# Save the ranked recommendations

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/capstone_ranked_recommendations.csv"

recommendations[columns_to_show].head(100).to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: work/outputs/capstone_ranked_recommendations.csv


In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
2,impressions_90d,0.269407
3,avg_position,0.250209
5,word_count,0.162847
0,content_age_days,0.162195
4,ctr,0.113559
1,days_since_last_update,0.041784


## Conclusion

This capstone demonstrates a content-refresh ranking workflow using anonymized search data.

The model combines several observable signals to prioritize pages for human review. The baseline provides a simple comparison point, while the model allows multiple signals to be considered together.

The results are measured decision-support findings from the available dataset. They should not be interpreted as causal proof or as a prediction of Google's ranking algorithm.

The final ranked queue is intended to help a content team decide where to investigate first.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


# 10. Week-8 Showcase: 5-Minute Demo Outline

## Question — 45 seconds

**Which webpages should be reviewed first for content refresh based on their existing search performance and content signals?**

The practical FlyRank content problem is that a content team may have many webpages to review, but limited time and resources. This project helps create a prioritized review queue so the team can decide where to investigate first.

## Method — 1 minute

I used an anonymized dataset containing 30,000 webpage observations.

The analysis used six observable signals:

- Content age
- Days since last update
- Search impressions
- Average search position
- CTR
- Word count

I created a declining-trend target and excluded outcome-revealing fields such as `trend_pct` to reduce target leakage.

I first created a simple freshness-and-visibility baseline and then trained a Random Forest model.

The model and baseline were evaluated using the same client-holdout test set.

## One Chart — 1 minute

The main chart compares the Random Forest with the baseline using:

- Precision@20
- Precision@50

The chart shows how effectively each approach places declining pages near the top of the content-review queue.

## One Honest Result — 1 minute

The Random Forest provides a learned ranking that can be compared directly with the simple baseline on the same held-out data.

The result should be interpreted as measured decision support within this dataset. It does not prove that the model causes traffic changes and it does not predict Google's ranking algorithm.

## One Recommendation — 1 minute

Use the final ranked queue to decide which webpages should be investigated first.

Pages with higher `refresh_score` should receive earlier human review, but the content team should check the actual page, search intent, content quality, competitors, and business context before making an update.

### Closing

The main takeaway is that the value of the project is not simply the machine-learning model. It is the complete workflow:

**content problem → data → leakage check → baseline → model → honest evaluation → ranked recommendations → human review**

# 11. Shareable Cuts

## Social Post

I built a machine-learning workflow to help prioritize webpages for content refresh using anonymized search and content signals.

The workflow starts with a simple freshness-and-visibility baseline, checks for target leakage, then compares it with a Random Forest model using client-holdout validation.

Instead of trying to predict Google's algorithm, the project focuses on a practical content-team question: **which pages should we investigate first?**

The final output is a ranked review queue that supports human content decisions.

#MachineLearning #DataScience #SEO #ContentStrategy #AI

---

## Employer-Facing Summary

I built a content-refresh prioritization workflow using an anonymized dataset of 30,000 webpages and six search/content signals: impressions, average position, CTR, word count, content age, and days since last update. I compared a simple freshness-and-visibility baseline with a Random Forest model using client-holdout validation and Precision@20 and Precision@50. The analysis showed how machine learning can support a ranked content-review queue while avoiding leakage and avoiding unsupported claims about causality or Google's ranking algorithm.